In [ ]:
import numpy as np
import matplotlib.pylab as plt
import matplotlib.colors as colors
import torch

In [ ]:
from skimage.metrics import peak_signal_noise_ratio

In [ ]:
print(torch.__version__)
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
device

In [ ]:
import time

In [ ]:
import sys
sys.path.insert(0, '../code') 
from network import UNet
from model_loader_func import load_UNet
#from plotting_func import plot_many_denoised
#from quality_metrics_func import calc_psnr, im_set_corr, normalized_distance_np
from dataloader_func import add_noise_torch

%matplotlib inline

In [ ]:
## Load pretraiend denoisers 

denoisers_face = {}
# denoisers in group A are trained on one partision of the data which is non-overlapping with partision B
groups = ['A'] 
swap = False
training_data_name = 'sdss'

Ns = [100,100000] #size of the training dataset 

for group in groups: 
    print('loading group ' , group )
    denoisers_face[group] = {}
    if group == 'B': 
        swap = True
    for N in Ns:       
        start_time_total = time.time()        
        try: 
            denoisers_face[group][N] = load_UNet(
                           base_path = '../denoisers/UNet',
                           training_data_name= training_data_name, 
                           training_noise='0to255',
                           RF=90,
                           set_size=N, 
                           swap=swap);
        except FileNotFoundError: 
            pass 
        print("--- %s seconds ---" % (round(time.time() - start_time_total)))


In [ ]:
train_all = torch.load('../../datasets/sdss_train_no_repeats_64x64.pt')

In [ ]:
train_all.shape

In [ ]:
n=100_005 # peak up an image never seen by any models (A or B)
#n=0 # seen during any models of type A
im = train_all[n:n+1] 

In [ ]:
im = im.to("cpu")

In [ ]:
### denoise the same test image with different levels of noise using denoisers in the same group 

all_denoised = {}
all_noisy = {}
sigmas = torch.tensor([0,10,50,150, 255])
for N in Ns:  
    all_denoised[N] = {}
    all_noisy[N] = {}
    torch.manual_seed(42)  # noisy image will be shared among the denoisers
    with torch.no_grad():
        for s in sigmas:
            s = np.round(s.item())
            noisy , _ = add_noise_torch( im , s)
            all_noisy[N][s] = noisy.cpu().numpy()
            denoised_im = noisy.to(device) - denoisers_face['A'][N](noisy.to(device)).detach()
            all_denoised[N][np.round(s/255, 3)] = denoised_im.cpu().numpy()

In [ ]:
for k in all_noisy[100].keys():
    print(np.alltrue(all_noisy[100][k]==all_noisy[100_000][k]))

In [ ]:
SMALL_SIZE = 24
MEDIUM_SIZE = 24
BIGGER_SIZE = 28

plt.rc('font', size=SMALL_SIZE)          # controls default text sizes
plt.rc('axes', titlesize=MEDIUM_SIZE)     # fontsize of the axes title
plt.rc('axes', labelsize=MEDIUM_SIZE)    # fontsize of the x and y labels
plt.rc('xtick', labelsize=SMALL_SIZE)    # fontsize of the tick labels
plt.rc('ytick', labelsize=SMALL_SIZE)    # fontsize of the tick labels
plt.rc('legend', fontsize=SMALL_SIZE)    # legend fontsize
plt.rc('figure', titlesize=BIGGER_SIZE)

def plot_denoised(
    x,        # clean image
    y,        # noisy image
    x_hat_1,  # denoiser 1
    x_hat_2,  # denoiser 2 
    device,
    label_den=["denoiser 1","denoiser 2"],
    suptitle=None,
    vmin=None,
    vmax=None,
    im_size=3,
    n_columns=None,
):

    x = x.numpy().squeeze()

    ############ plot noisy images
    if n_columns is None:
        n_columns = len(y)
        
    n_rows=3
    fig, axs = plt.subplots(n_rows ,n_columns, figsize = ( im_size*n_columns, n_rows*im_size ) )

    
    im_labels = [key for key,im in y.items() ]

    for i in range(len(y)):
        y[im_labels[i]] = y[im_labels[i]].squeeze()
        


    for i in range(len(y)):
        axs[0,i].imshow(y[im_labels[i]], 'gray',vmin=vmin, vmax = vmax)
        if i==0:
            axs[0,i].set_title("original")
            axs[0,i].set_ylabel("noisy input")
        else:
            axs[0,i].set_title( 'PSNR '+ str(round(peak_signal_noise_ratio(x,y[im_labels[i]], 
                                                                           data_range=1),3)))


    ############ plot denoised images
    im_labels = [key for key,im in x_hat_1.items() ]
    for i in range(len(x_hat_1)):
        x_hat_1[im_labels[i]] = x_hat_1[im_labels[i]].squeeze()
        axs[1,i].imshow(x_hat_1[im_labels[i]], 'gray',vmin=vmin, vmax = vmax)
        axs[1,i].set_title(
                    '\n PSNR '+ str(round(peak_signal_noise_ratio(x,x_hat_1[im_labels[i]], 
                                                                  data_range=1),2)))

        x_hat_2[im_labels[i]] = x_hat_2[im_labels[i]].squeeze()
        axs[2,i].imshow(x_hat_2[im_labels[i]], 'gray',vmin=vmin, vmax = vmax)
        axs[2,i].set_title(
                    '\n PSNR '+ str(round(peak_signal_noise_ratio(x,x_hat_2[im_labels[i]], 
                                                                  data_range=1),2)))
        if i==0:
            axs[1,i].set_ylabel(label_den[0])
            axs[2,i].set_ylabel(label_den[1])

    
    # polish
    axs = axs.ravel()
    for i in range(len(axs)):
        axs[i].set_xticks([])
        axs[i].set_yticks([])

    plt.subplots_adjust(left=None, bottom=None, right=None, top=None, wspace=0, hspace=0.2)

    if suptitle is not None:
        fig.suptitle(suptitle)
    
    #plt.tight_layout()

In [ ]:
plot_denoised(im, all_noisy[100], all_denoised[100],all_denoised[100000]  ,device="cpu", 
                   suptitle = "Train image" if n==0 else "Test image", 
                  label_den = ["N=100", "N=100,000"],
                   vmin=None, vmax=None, im_size=3
                 )
tag="train" if n==0 else "test"
plt.savefig("UNet_denoising_perf_"+tag+".pdf",bbox_inches='tight', pad_inches=0.1)